[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zhangdongming0607/ai-memory-course/blob/main/08-streaming-sse.ipynb)

> 点击上方按钮，在 Google Colab 中打开本课，可以直接运行代码，无需本地环境。

# 第 8 课：流式输出 — SSE 与打字机效果

在前面的课程中，我们每次调用 AI 都是「等它想完再一口气返回」。但实际产品中，ChatGPT、通义千问这些产品都是**一个字一个字蹦出来**的。

这节课我们来搞清楚：
1. 为什么需要流式输出
2. SSE 协议是什么
3. 如何用 OpenAI API 实现流式输出
4. 前端怎么接收（JS/TS）
5. 我们系统的 SSE 协议设计

---
## 8.1 为什么需要流式输出

AI 大模型生成一段回复，通常需要 **3-10 秒**（取决于回复长度和模型大小）。

有两种方式把结果给到用户：

| 方式 | 体验 | 类比 |
|------|------|------|
| **非流式** | 用户盯着空白等 10 秒，然后「啪」一下全出来 | 下载完视频再播放 |
| **流式** | 一个字一个字蹦出来，像有人在打字 | 边下边播（流媒体） |

对前端开发者来说，这和**视频流**的道理一样：
- 非流式 = 把整个 mp4 下载完再播放
- 流式 = 边下载边播放（HLS / DASH）

用户感知的「首字延迟」从 10 秒降到几百毫秒，体验天差地别。

In [ ]:
import time

# 模拟非流式：等待 3 秒后一次性输出
print("=== 非流式模式 ===")
print("用户发送问题...")
start = time.time()
time.sleep(3)  # 模拟 AI 思考
response = "流式输出能显著提升用户体验，让用户感觉 AI 在实时思考。"
print(f"等待 {time.time() - start:.1f} 秒后，一次性看到：")
print(response)

In [ ]:
import time
import sys

# 模拟流式：逐字输出
print("=== 流式模式 ===")
print("用户发送问题...")
time.sleep(0.3)  # 模拟首字延迟
response = "流式输出能显著提升用户体验，让用户感觉 AI 在实时思考。"
print("开始收到回复：")
for char in response:
    print(char, end="", flush=True)
    time.sleep(0.05)  # 每个字间隔 50ms
print()  # 换行
print("\n看到区别了吗？流式模式下，用户几乎立刻就能看到内容开始出现。")

---
## 8.2 SSE 协议入门

### 什么是 SSE？

**SSE = Server-Sent Events**，是 HTML5 规范的一部分。它允许服务器**单向**地向客户端推送数据。

### SSE vs WebSocket

| 特性 | SSE | WebSocket |
|------|-----|-----------|
| 通信方向 | 单向（服务器 -> 客户端） | 双向 |
| 协议 | HTTP | 独立协议（ws://） |
| 数据格式 | 纯文本 | 文本或二进制 |
| 自动重连 | 内置支持 | 需要手动实现 |
| 浏览器支持 | 原生支持（EventSource） | 原生支持 |
| 适用场景 | 服务器推送（通知、流式AI） | 实时双向通信（聊天室、游戏） |
| 复杂度 | 简单 | 较复杂 |

**为什么 AI 流式输出用 SSE 而不是 WebSocket？**
- AI 对话是「请求-响应」模式，服务器单向推送就够了
- SSE 基于 HTTP，更简单，不需要额外的协议升级
- OpenAI、Anthropic、各大模型厂商都用 SSE

### SSE 数据格式

SSE 的数据格式非常简单，就是纯文本：

```
event: message
data: {"content": "你"}

event: message
data: {"content": "好"}

event: message
data: {"content": "！"}

event: done
data: [DONE]
```

关键规则：
- 每个事件由 `event:` 和 `data:` 行组成
- 事件之间用**空行**（`\n\n`）分隔
- `data:` 后面是实际数据（通常是 JSON）
- 响应头必须是 `Content-Type: text/event-stream`

In [ ]:
# 手动构造一段 SSE 格式的数据，帮你理解格式

def format_sse_event(event: str, data: str) -> str:
    """构造一个 SSE 事件字符串"""
    return f"event: {event}\ndata: {data}\n\n"

# 模拟一段 SSE 流
events = [
    ("message", '{"content": "你"}'),
    ("message", '{"content": "好"}'),
    ("message", '{"content": "，"}'),
    ("message", '{"content": "世"}'),
    ("message", '{"content": "界"}'),
    ("done", "[DONE]"),
]

print("=== 这就是 SSE 在网络传输中的原始格式 ===")
print()
for event, data in events:
    sse_text = format_sse_event(event, data)
    # 用 repr() 显示换行符，方便理解
    print(f"原始文本: {repr(sse_text)}")
    print(f"实际显示:")
    print(sse_text)

In [ ]:
import json

def parse_sse_stream(raw_text: str) -> list[dict]:
    """解析 SSE 原始文本，返回事件列表"""
    events = []
    current_event = {}
    
    for line in raw_text.split("\n"):
        line = line.strip()
        if line.startswith("event:"):
            current_event["event"] = line[len("event:"):].strip()
        elif line.startswith("data:"):
            current_event["data"] = line[len("data:"):].strip()
        elif line == "" and current_event:
            events.append(current_event)
            current_event = {}
    
    # 别忘了最后一个事件
    if current_event:
        events.append(current_event)
    
    return events

# 测试解析
raw_sse = """event: message
data: {"content": "你"}

event: message
data: {"content": "好"}

event: done
data: [DONE]
"""

parsed = parse_sse_stream(raw_sse)
print("解析结果：")
for evt in parsed:
    print(f"  事件类型: {evt['event']}, 数据: {evt['data']}")

# 拼接内容
full_text = ""
for evt in parsed:
    if evt["event"] == "message":
        content = json.loads(evt["data"])["content"]
        full_text += content
print(f"\n拼接后的完整文本: {full_text}")

---
## 8.3 用 OpenAI API 体验流式输出

OpenAI 的 API 只需要加一个参数 `stream=True` 就能开启流式输出。

我们来对比流式和非流式的区别。

In [ ]:
# 先安装依赖（如果还没装的话）
# !pip install openai

from openai import OpenAI
import time
import os

# 初始化客户端（请确保设置了 OPENAI_API_KEY 环境变量）
client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY", "your-api-key-here"),
    # 如果用国内代理，取消下面这行的注释并填写你的代理地址
    # base_url="https://your-proxy.com/v1",
)

prompt = "用 3 句话解释什么是 SSE（Server-Sent Events）"

# === 非流式调用 ===
print("=== 非流式调用 ===")
start = time.time()
response = client.chat.completions.create(
    model="gpt-3.5-turbo",
    messages=[{"role": "user", "content": prompt}],
    stream=False,  # 默认就是 False
)
elapsed = time.time() - start
print(f"耗时: {elapsed:.2f} 秒")
print(f"回复: {response.choices[0].message.content}")
print()

In [ ]:
import time
import sys

# === 流式调用 ===
print("=== 流式调用 ===")
start = time.time()
first_token_time = None

stream = client.chat.completions.create(
    model="gpt-3.5-turbo",
    messages=[{"role": "user", "content": prompt}],
    stream=True,  # 开启流式！
)

full_response = ""
for chunk in stream:
    # 每个 chunk 是一小段内容
    if chunk.choices[0].delta.content is not None:
        token = chunk.choices[0].delta.content
        if first_token_time is None:
            first_token_time = time.time()
        full_response += token
        print(token, end="", flush=True)  # 逐字打印，模拟打字机

elapsed = time.time() - start
first_token_delay = first_token_time - start if first_token_time else 0

print(f"\n\n首字延迟: {first_token_delay:.2f} 秒")
print(f"总耗时: {elapsed:.2f} 秒")
print(f"\n关键指标对比：")
print(f"  非流式 - 用户要等全部生成完才能看到，体验上就是干等")
print(f"  流式   - 首字 {first_token_delay:.2f} 秒就出来了，后面逐字出现，体验流畅")

In [ ]:
# 如果没有 OpenAI API Key，用这个模拟版本来体验效果

import time
import sys

def fake_stream(text: str, chunk_size: int = 2, delay: float = 0.05):
    """模拟流式输出：将文本分成小块，逐块 yield"""
    for i in range(0, len(text), chunk_size):
        time.sleep(delay)
        yield text[i:i + chunk_size]

simulated_response = (
    "SSE（Server-Sent Events）是一种基于 HTTP 的服务器推送技术，"
    "允许服务器通过持久连接向客户端单向发送实时数据更新。"
    "它使用简单的文本格式（event/data），浏览器通过 EventSource API 原生支持，"
    "非常适合 AI 流式输出、实时通知等场景。"
)

print("=== 模拟流式输出（无需 API Key） ===")
print()

start = time.time()
first_chunk_time = None
full_text = ""

for chunk in fake_stream(simulated_response):
    if first_chunk_time is None:
        first_chunk_time = time.time()
    full_text += chunk
    print(chunk, end="", flush=True)

print(f"\n\n首字延迟: {first_chunk_time - start:.3f} 秒")
print(f"总耗时: {time.time() - start:.2f} 秒")

---
## 8.4 前端怎么接（JavaScript/TypeScript 示例）

作为前端开发者，你需要知道怎么在浏览器端接收 SSE 流。有两种主流方式：

### 方式 1：EventSource（简单但不够灵活）

```typescript
// EventSource 是浏览器原生 API，使用最简单
// 但它只支持 GET 请求，不能自定义 header

const eventSource = new EventSource('/api/chat/stream?message=你好');

let fullResponse = '';

// 监听消息事件
eventSource.addEventListener('message', (event) => {
  const data = JSON.parse(event.data);
  fullResponse += data.content;
  
  // 更新 UI
  document.getElementById('response').textContent = fullResponse;
});

// 监听完成事件
eventSource.addEventListener('done', (event) => {
  console.log('生成完毕');
  eventSource.close();  // 必须手动关闭！
});

// 监听错误
eventSource.addEventListener('error', (event) => {
  console.error('SSE 连接出错:', event);
  eventSource.close();
});
```

**EventSource 的局限：**
- 只能发 GET 请求（聊天消息通常用 POST）
- 不能自定义请求头（比如 Authorization）
- 不能发送请求体

### 方式 2：fetch + ReadableStream（推荐）

```typescript
// 这是实际项目中最常用的方式
// 支持 POST、自定义 header、请求体，完全可控

async function chatStream(message: string) {
  const response = await fetch('/api/chat/stream', {
    method: 'POST',
    headers: {
      'Content-Type': 'application/json',
      'Authorization': 'Bearer your-token',
    },
    body: JSON.stringify({ message }),
  });

  if (!response.ok) {
    throw new Error(`HTTP error! status: ${response.status}`);
  }

  // 关键：拿到 ReadableStream
  const reader = response.body!.getReader();
  const decoder = new TextDecoder();
  let fullResponse = '';
  let buffer = '';  // SSE 数据可能跨 chunk，需要缓冲区

  while (true) {
    const { done, value } = await reader.read();
    if (done) break;

    // 解码二进制为文本
    buffer += decoder.decode(value, { stream: true });

    // 按 \n\n 分割事件
    const events = buffer.split('\n\n');
    buffer = events.pop() || '';  // 最后一个可能不完整，留在缓冲区

    for (const eventStr of events) {
      if (!eventStr.trim()) continue;

      // 解析每个 SSE 事件
      const lines = eventStr.split('\n');
      let eventType = 'message';
      let data = '';

      for (const line of lines) {
        if (line.startsWith('event:')) {
          eventType = line.slice(6).trim();
        } else if (line.startsWith('data:')) {
          data = line.slice(5).trim();
        }
      }

      // 根据事件类型处理
      if (eventType === 'message' && data !== '[DONE]') {
        const parsed = JSON.parse(data);
        fullResponse += parsed.content;
        // 更新 UI - 打字机效果
        updateUI(fullResponse);
      } else if (data === '[DONE]') {
        console.log('生成完毕，完整回复:', fullResponse);
      }
    }
  }
}

function updateUI(text: string) {
  const el = document.getElementById('response');
  if (el) el.textContent = text;
}

// 使用
chatStream('帮我解释什么是 SSE');
```

### 两种方式对比

| 特性 | EventSource | fetch + ReadableStream |
|------|------------|------------------------|
| 请求方法 | 仅 GET | 任意（GET/POST/...） |
| 自定义 Header | 不支持 | 支持 |
| 请求体 | 不支持 | 支持 |
| 自动重连 | 内置 | 需手动实现 |
| 事件类型解析 | 自动 | 需手动解析 |
| 实际项目推荐 | 简单场景 | **复杂场景（推荐）** |

In [ ]:
# 用 Python 模拟前端 fetch + ReadableStream 的处理过程
# 帮助理解前端代码中的「缓冲区」和「按 \n\n 分割」逻辑

def simulate_network_chunks(sse_data: str, chunk_size: int = 30) -> list[bytes]:
    """模拟网络传输：把 SSE 数据切成不规则的字节块（就像 TCP 包一样）"""
    raw = sse_data.encode("utf-8")
    chunks = []
    for i in range(0, len(raw), chunk_size):
        chunks.append(raw[i:i + chunk_size])
    return chunks

# 模拟服务器返回的完整 SSE 流
sse_payload = (
    "event: message\ndata: {\"content\": \"Hello\"}\n\n"
    "event: message\ndata: {\"content\": \" World\"}\n\n"
    "event: message\ndata: {\"content\": \"!\"}\n\n"
    "event: done\ndata: [DONE]\n\n"
)

# 模拟网络切割（注意：切割位置是随机的，可能切在事件中间）
chunks = simulate_network_chunks(sse_payload, chunk_size=25)
print("网络传来的原始字节块：")
for i, chunk in enumerate(chunks):
    print(f"  chunk {i}: {chunk}")

print("\n=== 像前端一样处理这些 chunk ===")

buffer = ""  # 前端代码中的缓冲区
full_response = ""

for chunk in chunks:
    # 解码（对应 TextDecoder.decode）
    buffer += chunk.decode("utf-8")
    
    # 按 \n\n 分割
    parts = buffer.split("\n\n")
    buffer = parts[-1]  # 最后一部分可能不完整
    
    for event_str in parts[:-1]:
        if not event_str.strip():
            continue
        # 解析事件
        event_type = "message"
        data = ""
        for line in event_str.split("\n"):
            if line.startswith("event:"):
                event_type = line[6:].strip()
            elif line.startswith("data:"):
                data = line[5:].strip()
        
        if event_type == "message" and data != "[DONE]":
            content = json.loads(data)["content"]
            full_response += content
            print(f"  收到: {repr(content)}  ->  当前文本: {repr(full_response)}")
        elif event_type == "done":
            print(f"  收到 done 事件，生成完毕")

print(f"\n最终完整文本: {full_response}")

---
## 8.5 我们系统的 SSE 协议

在实际的 AI 记忆系统中，我们定义了 **5 种 SSE 事件类型**，覆盖一次 AI 回复的完整生命周期：

| 事件类型 | 含义 | 前端处理 |
|----------|------|----------|
| `ack` | 服务器确认收到请求，返回 run_id | 保存 run_id（用于取消生成） |
| `delta` | 一小段生成的文本 | 追加到显示区域（打字机效果） |
| `done` | 生成完毕 | 结束 loading 状态 |
| `error` | 出错了 | 显示错误提示 |
| `usage` | token 用量统计 | 显示消耗或用于计费 |

### 一次完整的 SSE 交互流程

```
前端 POST /api/chat/stream {message: "你好"}
  |
  v
服务器返回 SSE 流：
  event: ack
  data: {"run_id": "run_abc123"}
  
  event: delta
  data: {"content": "你"}
  
  event: delta
  data: {"content": "好"}
  
  event: delta
  data: {"content": "！"}
  
  event: usage
  data: {"prompt_tokens": 10, "completion_tokens": 3, "total_tokens": 13}
  
  event: done
  data: {"run_id": "run_abc123"}
```

### 停止生成

用户点击「停止生成」按钮时，前端用 `ack` 拿到的 `run_id` 调用取消接口：

```
POST /api/chat/cancel
Body: {"run_id": "run_abc123"}
```

In [ ]:
import json
import time
import uuid
from dataclasses import dataclass, field
from typing import Generator


@dataclass
class SSEEvent:
    """一个 SSE 事件"""
    event: str
    data: dict | str

    def to_sse_string(self) -> str:
        data_str = json.dumps(self.data, ensure_ascii=False) if isinstance(self.data, dict) else self.data
        return f"event: {self.event}\ndata: {data_str}\n\n"


class ChatStreamSimulator:
    """模拟我们系统的 SSE 聊天流"""

    def __init__(self):
        self._cancelled_runs: set[str] = set()

    def stream_chat(self, message: str) -> Generator[SSEEvent, None, None]:
        """模拟服务器端的流式生成过程"""
        run_id = f"run_{uuid.uuid4().hex[:8]}"

        # 1. 发送 ack
        yield SSEEvent(event="ack", data={"run_id": run_id})

        # 2. 模拟 AI 生成
        fake_response = f"收到你的消息：「{message}」。这是一个模拟的 AI 回复，用来演示我们的 SSE 协议。"
        prompt_tokens = len(message) * 2  # 简化计算
        completion_tokens = 0

        for char in fake_response:
            # 检查是否被取消
            if run_id in self._cancelled_runs:
                yield SSEEvent(event="error", data={"code": "CANCELLED", "message": "生成已被用户取消"})
                return

            time.sleep(0.03)  # 模拟生成延迟
            completion_tokens += 1
            yield SSEEvent(event="delta", data={"content": char})

        # 3. 发送 usage
        yield SSEEvent(event="usage", data={
            "prompt_tokens": prompt_tokens,
            "completion_tokens": completion_tokens,
            "total_tokens": prompt_tokens + completion_tokens,
        })

        # 4. 发送 done
        yield SSEEvent(event="done", data={"run_id": run_id})

    def cancel(self, run_id: str):
        """取消一个正在进行的生成"""
        self._cancelled_runs.add(run_id)
        print(f"[服务器] 收到取消请求: {run_id}")


# 创建模拟器
simulator = ChatStreamSimulator()
print("ChatStreamSimulator 已创建，接下来我们用它来模拟完整的 SSE 交互")

In [ ]:
import sys

# 模拟前端处理 SSE 流

def frontend_handle_stream(stream: Generator):
    """模拟前端接收和处理 SSE 事件的逻辑"""
    run_id = None
    full_text = ""

    for event in stream:
        if event.event == "ack":
            run_id = event.data["run_id"]
            print(f"[前端] 收到 ack，run_id = {run_id}")
            print(f"[前端] 开始显示打字机效果...")
            print(f"[前端] ", end="")

        elif event.event == "delta":
            content = event.data["content"]
            full_text += content
            print(content, end="", flush=True)  # 打字机效果

        elif event.event == "usage":
            print(f"\n[前端] Token 用量 - 输入: {event.data['prompt_tokens']}, "
                  f"输出: {event.data['completion_tokens']}, "
                  f"总计: {event.data['total_tokens']}")

        elif event.event == "done":
            print(f"[前端] 生成完毕! run_id = {event.data['run_id']}")

        elif event.event == "error":
            print(f"\n[前端] 错误: {event.data['message']}")

    return full_text


# 正常流程
print("=" * 50)
print("模拟一次完整的 SSE 对话流程")
print("=" * 50)

stream = simulator.stream_chat("什么是 AI 记忆系统？")
result = frontend_handle_stream(stream)

print(f"\n\n完整回复: {result}")

In [ ]:
import threading

# 模拟「停止生成」功能

print("=" * 50)
print("模拟用户点击「停止生成」")
print("=" * 50)

# 创建一个新的模拟器
sim = ChatStreamSimulator()
captured_run_id = None

stream = sim.stream_chat("给我讲一个很长很长的故事")

full_text = ""
for i, event in enumerate(stream):
    if event.event == "ack":
        captured_run_id = event.data["run_id"]
        print(f"[前端] 收到 ack，run_id = {captured_run_id}")
        print(f"[前端] ", end="")

    elif event.event == "delta":
        content = event.data["content"]
        full_text += content
        print(content, end="", flush=True)

        # 收到 10 个字后，模拟用户点击停止
        if len(full_text) == 10:
            print("\n[前端] 用户点击了「停止生成」按钮!")
            sim.cancel(captured_run_id)

    elif event.event == "error":
        print(f"[前端] 生成已停止: {event.data['message']}")
        break

    elif event.event == "done":
        print(f"\n[前端] 生成完毕")

print(f"\n已收到的文本: {full_text}")

In [ ]:
# 把我们的 SSE 事件转成真实的 SSE 文本格式，看看网络上传输的到底是什么

print("=== 我们协议的事件在网络上长这样 ===")
print()

sample_events = [
    SSEEvent(event="ack", data={"run_id": "run_abc123"}),
    SSEEvent(event="delta", data={"content": "你"}),
    SSEEvent(event="delta", data={"content": "好"}),
    SSEEvent(event="delta", data={"content": "！"}),
    SSEEvent(event="usage", data={"prompt_tokens": 10, "completion_tokens": 3, "total_tokens": 13}),
    SSEEvent(event="done", data={"run_id": "run_abc123"}),
]

full_sse_text = ""
for evt in sample_events:
    sse_str = evt.to_sse_string()
    full_sse_text += sse_str
    print(sse_str, end="")

print("---")
print(f"总共 {len(full_sse_text)} 字节")
print("\n这就是前端 fetch 拿到的原始文本，前端需要自己解析这些内容。")

---
## 8.6 本课小结

### 核心概念

| 概念 | 要点 |
|------|------|
| 流式输出 | 把 AI 回复拆成小块逐个推送，用户体验从「干等」变成「看打字」 |
| SSE 协议 | 基于 HTTP 的服务器单向推送，格式简单（event + data + 空行分隔） |
| OpenAI stream | 加 `stream=True` 即可，返回的每个 chunk 包含一小段 delta |
| 前端接收 | EventSource（简单）或 fetch + ReadableStream（灵活，推荐） |
| 我们的协议 | 5 种事件：ack / delta / done / error / usage |
| 停止生成 | 通过 ack 返回的 run_id + cancel API 实现 |

### 和我们系统的对应关系

在我们的 AI 记忆系统中：

- **keyboard_stream**：用于键盘输入联想，使用 SSE 流式返回补全建议
- **chat stream**：用于对话场景，使用本课介绍的 5 种事件协议

两者本质上都是 SSE，只是事件类型和数据结构不同。

### 下一课预告

下一课我们将进入**记忆的存储与检索**，看看 AI 是怎么「记住」用户说过的话的。

In [ ]:
# 最后的练习：自己动手试一试

print("=== 课后练习 ===")
print()
print("练习 1: 修改 fake_stream 函数，让它每次返回一个完整的词而不是固定长度的块")
print("练习 2: 给 ChatStreamSimulator 添加一个 'typing' 事件，在开始生成前告诉前端 AI 正在思考")
print("练习 3: 用 fetch + ReadableStream 的思路，在前端实现一个真正能用的 SSE 客户端")
print()
print("提示：练习 3 可以配合一个简单的 FastAPI/Flask 后端来测试")